# 第3章 演習：Tool呼び出しとMemory (Checkpointer) の実装

## この演習について

`# TODO:` のある箇所にコードを記述して、ツール付きの記憶するエージェントを完成させてください。

---

In [ ]:
%pip install -qU langchain langchain-openai langgraph pydantic

In [ ]:
import os
from google.colab import userdata

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "chap03-exercise"
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print("環境変数の設定が完了しました")

## 課題1: カスタムツールを定義する

**目標:** `@tool` デコレータを使って2つのツールを定義してください。

**ヒント:**
- `from langchain.tools import tool` でインポート
- 関数に `@tool` デコレータを付け、**型ヒント付きの引数** と **docstring（説明文）** を必ず書くこと
- docstring の `Args:` セクションに引数の説明を書くと、LLM の理解度が上がります

In [ ]:
# TODO: tool をインポートしてください
# from ??? import ???

# TODO: 都市の天気を返すツール get_weather を定義してください
# - 引数: city: str（天気を調べる都市名）
# - 固定値のダミーデータ（辞書）で東京・大阪・名古屋の天気を返す
# @tool
# def get_weather(city: str) -> str:
#     """..."""
#     ...


# TODO: 計算式を評価するツール calculate を定義してください
# - 引数: expression: str（計算式の文字列、例: '100 + 200'）
# - eval() を使って計算結果を返す（try/except でエラー処理も忘れずに）
# @tool
# def calculate(expression: str) -> str:
#     """..."""
#     ...


# ツールの動作確認
# print(get_weather.invoke({"city": "東京"}))
# print(calculate.invoke({"expression": "(100 + 200) * 3"}))

## 課題2: ツール付きエージェントを作成する

**目標:** `create_agent` に上で定義したツールを渡してエージェントを作成し、
「東京の天気を教えて。また 123 × 456 を計算して」という質問を投げてください。

**ヒント:**
- `tools=[get_weather, calculate]` のようにリストで渡す
- `agent.invoke()` の戻り値の最後のメッセージが最終回答

In [ ]:
from langchain.agents import create_agent

# TODO: ツール付きエージェントを作成してください
# agent_with_tools = create_agent(
#     model=???,
#     tools=???,         # 課題1で定義した2つのツールを渡す
#     system_prompt=???, # 適切なシステムプロンプトを記述
# )

# TODO: エージェントに「東京の天気」と「123 × 456 の計算」を同時に質問してください
# result = agent_with_tools.invoke({
#     "messages": [{"role": "user", "content": ???}]
# })
# print(result["messages"][-1].content)

## 課題3: Checkpointer（Memory）を追加して会話記憶を実装する

**目標:** `InMemorySaver` を使い、同じ `thread_id` で2回会話して、
2回目の返答が1回目の会話を踏まえていることを確認してください。

**ヒント:**
- `from langgraph.checkpoint.memory import InMemorySaver` でインポート
- `create_agent` の引数に `checkpointer=InMemorySaver()` を追加
- `agent.invoke()` に `config={"configurable": {"thread_id": "xxx"}}` を渡す

In [ ]:
# TODO: InMemorySaver をインポートしてください
# from ??? import ???

# TODO: Checkpointer を使うエージェントを作成してください
# agent_with_memory = create_agent(
#     model="openai:gpt-4o",
#     tools=[get_weather, calculate],
#     system_prompt="あなたは親切なAIアシスタントです。日本語で回答してください。",
#     checkpointer=???,  # InMemorySaver のインスタンスを渡す
# )

# TODO: config に thread_id を設定してください
# config = ???

# --- ターン1: 東京の天気を質問 ---
# result1 = agent_with_memory.invoke(
#     {"messages": [{"role": "user", "content": "東京の天気を教えてください"}]},
#     config=config,
# )
# print("[ターン1]", result1["messages"][-1].content)

# --- ターン2: さっきの都市の気温を確認（前の会話を参照） ---
# result2 = agent_with_memory.invoke(
#     {"messages": [{"role": "user", "content": "さっき調べた都市の気温は何度でしたか？"}]},
#     config=config,  # 同じ thread_id を使って継続
# )
# print("[ターン2]", result2["messages"][-1].content)

---

## チャレンジ問題（余力があれば）

Pydantic BaseModel を使った `response_format` を試してみましょう。
以下のスキーマを使って、天気レポートを構造化出力させてください。

```python
from pydantic import BaseModel, Field

class WeatherReport(BaseModel):
    city: str = Field(description="都市名")
    condition: str = Field(description="天気の状態")
    temperature_celsius: float = Field(description="気温（摂氏）")
    recommendation: str = Field(description="外出に関するアドバイス")
```

正解は `exercise/solution/chap03_solution.ipynb` を確認してください。